In [1]:
from sklearn.model_selection import train_test_split, cross_val_predict, KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, roc_auc_score
from sklearn.metrics import confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from plotly.subplots import make_subplots
import plotly.graph_objs as go
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import RandomizedSearchCV
import shap
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_validate
import plotly.express as px
from catboost import CatBoostRegressor
from sklearn.dummy import DummyRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
from sklearn.compose import TransformedTargetRegressor

In [2]:
RANDOM_STATE = 12345

In [3]:
df = pd.read_csv("../data/diamonds.csv")

In [4]:
df = df.drop_duplicates()

In [5]:
x_precentile_99 = df['x'].quantile(0.99)
y_precentile_99 = df['y'].quantile(0.99)
z_precentile_99 = df['z'].quantile(0.99)
x_precentile_01 = df['x'].quantile(0.01)
y_precentile_01 = df['y'].quantile(0.01)
z_precentile_01 = df['z'].quantile(0.01)
df = df[df['x']<= x_precentile_99]
df = df[df['y']<= y_precentile_99]
df = df[df['z']<= z_precentile_99]
df = df[df['x']>= x_precentile_01]
df = df[df['y']>= y_precentile_01]
df = df[df['z']>= z_precentile_01]

In [6]:
X = df.drop('price', axis = 1)

In [7]:
y = df['price']

In [8]:
X = X.drop(columns = 'Unnamed: 0')

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = RANDOM_STATE)

In [10]:
num_cols = X_train.select_dtypes(include='number').columns.tolist()

In [11]:
cat_cols = X_train.select_dtypes(include='object').columns.tolist()

/var/folders/jt/pfrf2kqn59d1ph0418pyq4mr0000gn/T/ipykernel_48291/4180746908.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include='object').columns.tolist()


In [12]:
cat_cols = [col for col in cat_cols if col not in ['cut', 'color', 'clarity']]

In [13]:
ord_cols = ['cut', 'color', 'clarity']

In [14]:
cut_order = ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal']
color_order = ['J', 'I', 'H', 'G', 'F', 'E', 'D']
clarity_order = ['I1', 'SI2', 'SI1', 'VS2', 'VS1', 'VVS2', 'VVS1', 'IF']

In [15]:
preprocessor = ColumnTransformer([
    ('num', 'passthrough', num_cols),

    ('ord',
     OrdinalEncoder(
         categories=[
             cut_order,
             color_order,
             clarity_order
         ]
     ),
     ord_cols),

    ('cat',
     OneHotEncoder(handle_unknown='ignore'),
     cat_cols)
])

Функция оценки

In [16]:
def evaluate_pipeline(pipeline, X, y, cv):

    scoring = {
        "mae": "neg_mean_absolute_error",
        "rmse": "neg_root_mean_squared_error",
        "r2": "r2"
    }

    scores = cross_validate(
        pipeline,
        X,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    return {
        "mae": -scores["test_mae"].mean(),
        "rmse": -scores["test_rmse"].mean(),
        "r2": scores["test_r2"].mean()
    }

Эксперимент 1
Удаление признаков x, y, z

In [17]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

In [19]:
X_no_xyz = X_train.drop(columns=['x', 'y', 'z'])

cat_cols = X_no_xyz.select_dtypes(include="object").columns.tolist()
num_cols = X_no_xyz.select_dtypes(exclude="object").columns.tolist()

preprocessor_no_xyz = ColumnTransformer([
    ('num', 'passthrough', num_cols),  
    
    ('ord',
     OrdinalEncoder(
         categories=[cut_order, color_order, clarity_order],
         handle_unknown='use_encoded_value',
         unknown_value=-1
     ),
     ord_cols),
    
    ('cat',
     OneHotEncoder(handle_unknown='ignore', sparse_output=False),
     cat_cols)
], remainder='drop')

model_no_xyz = Pipeline([
    ("preprocessor", preprocessor_no_xyz),
    ("model", CatBoostRegressor(
        iterations=300,
        depth=6,
        learning_rate=0.1,
        verbose=0,
        random_state=RANDOM_STATE
    ))
])

results_no_xyz = evaluate_pipeline(
    model_no_xyz,
    X_no_xyz,
    y_train,
    cv
)

results_no_xyz

/var/folders/jt/pfrf2kqn59d1ph0418pyq4mr0000gn/T/ipykernel_48291/2608726562.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_no_xyz.select_dtypes(include="object").columns.tolist()


{'mae': np.float64(268.26887042795704),
 'rmse': np.float64(498.4943129014326),
 'r2': np.float64(0.9827701528897574)}

Вывод эксперимента: не стоит убирать признаки x, y, z, так как они улучшают качество

Эксперимент 2

Логорифмирование таргета

In [21]:
model_log_target = Pipeline([
    ("preprocessor", preprocessor),
    ("model", CatBoostRegressor(
        iterations=300,
        depth=6,
        learning_rate=0.1,
        verbose=0,
        random_state=RANDOM_STATE
    ))
])

results_log_target = evaluate_pipeline(
    model_log_target,
    X_train,
    y_train,
    cv
)

results_log_target

{'mae': np.float64(268.7205287963696),
 'rmse': np.float64(498.3287245703127),
 'r2': np.float64(0.9827846281007971)}

Вывод эксперимента 2: качество модели улучшилось.

Эксперимент 3

Создание дополнительных признаков на основе карата

In [23]:
X_fe = X_train.copy()

# 1. Объём
X_fe['volume'] = X_fe['x'] * X_fe['y'] * X_fe['z']

# 2. Карат на объём
X_fe['carat_per_volume'] = X_fe['carat'] / (X_fe['volume'] + 1e-8)

# 3. Отношение table к depth
X_fe['table_depth_ratio'] = X_fe['table'] / (X_fe['depth'] + 1e-8)

num_cols_fe = X_fe.select_dtypes(include='number').columns.tolist()
cat_cols_fe = X_fe.select_dtypes(include='object').columns.tolist()

ord_cols = ['cut', 'color', 'clarity']
cat_cols_fe = [col for col in cat_cols_fe if col not in ord_cols]

preprocessor_fe = ColumnTransformer([
    ('num', 'passthrough', num_cols_fe),
    
    ('ord',
     OrdinalEncoder(
         categories=[cut_order, color_order, clarity_order],
         handle_unknown='use_encoded_value',
         unknown_value=-1
     ),
     ord_cols),
    
    ('cat',
     OneHotEncoder(handle_unknown='ignore', sparse_output=False),
     cat_cols_fe)
], remainder='drop')

model_fe = Pipeline([
    ("preprocessor", preprocessor_fe),
    ("model", CatBoostRegressor(
        iterations=300,
        depth=6,
        learning_rate=0.1,
        verbose=0,
        random_state=RANDOM_STATE
    ))
])

results_fe = evaluate_pipeline(
    model_fe,
    X_fe,
    y_train,
    cv
)

results_fe

/var/folders/jt/pfrf2kqn59d1ph0418pyq4mr0000gn/T/ipykernel_48291/1866099348.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols_fe = X_fe.select_dtypes(include='object').columns.tolist()


{'mae': np.float64(266.89875947713483),
 'rmse': np.float64(495.81991223878447),
 'r2': np.float64(0.9829589126309702)}

Вывод эксперимента 3: Создание доп признаков улучшило качество модели

Обший вывод: только первый эксперимент не показал метрику выше